# Demo: Training a Neural Forecasting Model


In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.utils.likelihood_models import QuantileRegression

HORIZON = 12

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date").asfreq("MS")
subscribers = df["subscribers"]
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))

print(f"Data: {len(subscribers)} months")
print(f"Train: {len(train)}, Test: {len(test)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
subscribers.plot(ax=ax, color=UN["Black"])
ax.set_title("Monthly Subscribers", fontsize=18, fontweight="bold")
ax.set_ylabel("Subscribers", fontsize=16)
ax.axvline(pd.Timestamp("2023-12-01"), color=US["Red"], linestyle="--", alpha=0.5, label="Train/Test split")
ax.legend()
ax.tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()

## Training N-BEATS


In [ ]:
nbeats = NBEATSModel(
    input_chunk_length=24,
    output_chunk_length=12,
    n_epochs=50,
    likelihood=QuantileRegression(quantiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
    random_state=42,
    pl_trainer_kwargs={"enable_progress_bar": True},
)
nbeats.fit(train)
nbeats_fc = nbeats.predict(HORIZON, num_samples=100)

## ARIMA Baseline


In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

## Comparing


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ts.plot(ax=ax, label="Actual", color=UN["Black"])
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color=UB["Brand Blue"])
nbeats_fc.plot(ax=ax, label="N-BEATS", color=UB["Medium Blue"], low_quantile=0.05, high_quantile=0.95)
ax.set_ylabel("Subscribers", fontsize=16)
ax.set_title("Forecast Comparison", fontsize=18, fontweight="bold")
ax.legend(loc="upper left")
ax.tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()
